# 02 - Data Cleaning & Master Analytical Dataset Creation
## Handling Missing Values, Date Conversions, Status Validations & Merges

**Objective:**
Clean and merge all tables into a cohesive order-level analytical dataset for retention and cohort modeling.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


### 1. Load Raw Tables


In [ ]:
customers = pd.read_csv(RAW_DIR / 'olist_customers_dataset.csv')
orders = pd.read_csv(RAW_DIR / 'olist_orders_dataset.csv')
order_items = pd.read_csv(RAW_DIR / 'olist_order_items_dataset.csv')
payments = pd.read_csv(RAW_DIR / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(RAW_DIR / 'olist_order_reviews_dataset.csv')
products = pd.read_csv(RAW_DIR / 'olist_products_dataset.csv')
category_translation = pd.read_csv(RAW_DIR / 'product_category_name_translation.csv')
print('Datasets loaded successfully.')


### 2. Parse Datetime Fields with Dayfirst Parsing


In [ ]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], format='mixed', dayfirst=True)
    print(f'Converted {col} to datetime.')


### 3. Aggregate Order Items, Payments & Reviews per Order


In [ ]:
# 1. Order items aggregation
items_agg = order_items.groupby('order_id').agg(
    item_count=('order_item_id', 'count'),
    total_items_price=('price', 'sum'),
    total_freight_value=('freight_value', 'sum')
).reset_index()
items_agg['order_revenue'] = items_agg['total_items_price'] + items_agg['total_freight_value']

# 2. Payments aggregation
payments_agg = payments.groupby('order_id').agg(
    total_payment_value=('payment_value', 'sum'),
    payment_installments_max=('payment_installments', 'max'),
    primary_payment_type=('payment_type', lambda x: x.mode()[0] if not x.empty else 'unknown')
).reset_index()

# 3. Reviews deduplication (mean score per order)
reviews_agg = reviews.groupby('order_id').agg(
    review_score=('review_score', 'mean'),
    review_count=('review_id', 'count')
).reset_index()

print('Aggregations completed.')


### 4. Merge Master Order Analytical Dataset


In [ ]:
master = orders.merge(customers, on='customer_id', how='left')
master = master.merge(items_agg, on='order_id', how='left')
master = master.merge(payments_agg, on='order_id', how='left')
master = master.merge(reviews_agg, on='order_id', how='left')

# Add delivery performance features
master['delivery_time_days'] = (master['order_delivered_customer_date'] - master['order_purchase_timestamp']).dt.total_seconds() / 86400.0
master['delivery_delay_days'] = (master['order_delivered_customer_date'] - master['order_estimated_delivery_date']).dt.total_seconds() / 86400.0
master['is_delayed'] = np.where(master['delivery_delay_days'] > 0, 1, 0)
master['delivery_status'] = np.where(
    master['order_delivered_customer_date'].isna(), 'Not Delivered',
    np.where(master['delivery_delay_days'] > 0, 'Late', 'On-Time')
)

master.to_csv(PROCESSED_DIR / 'master_analytical_orders.csv', index=False)
print(f'Master dataset saved: {master.shape[0]:,} rows x {master.shape[1]} columns.')
